# Functional SlowHeat — benchmark Split MNIST class-incremental

Este notebook executa cinco tarefas sequenciais (`0/1`, `2/3`, ..., `8/9`) com um único MLP. A inferência não recebe task ID, classes futuras não participam da loss e todos os métodos usam a mesma inicialização, dados e sequência de minibatches.

O objetivo é medir simultaneamente **retenção** e **plasticidade**. Uma queda de forgetting acompanhada por queda maior de acurácia não é uma melhoria global.

## Preparação

Na raiz do repositório, instale uma vez as dependências: `python -m pip install -e '.[research]'`. Na primeira execução, o torchvision baixará o MNIST para `data/`.

In [ ]:
import os
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

FALLBACK_ROOT = Path(
    os.environ.get('DUAL_HEATER_ROOT', Path.home() / 'dual-heater-mlp-research')
).expanduser()
try:
    START = Path.cwd().resolve()
except FileNotFoundError:
    if not FALLBACK_ROOT.is_dir():
        raise RuntimeError(
            'O diretório do kernel foi removido e o repositório não foi encontrado em '
            f'{FALLBACK_ROOT}. Defina DUAL_HEATER_ROOT com o caminho correto.'
        )
    os.chdir(FALLBACK_ROOT)
    START = FALLBACK_ROOT.resolve()

ROOT = next(
    (candidate for candidate in (START, *START.parents)
     if (candidate / 'pyproject.toml').is_file()
     and (candidate / 'experiments').is_dir()),
    None,
)
if ROOT is None:
    if FALLBACK_ROOT.is_dir():
        ROOT = FALLBACK_ROOT.resolve()
    else:
        raise RuntimeError(
            'Raiz do projeto não encontrada. Defina DUAL_HEATER_ROOT antes de iniciar o Jupyter.'
        )
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from experiments.split_mnist import (
    SplitMNISTConfig,
    load_split_mnist,
    run_split_mnist,
    run_split_mnist_epoch_sweep,
    run_split_mnist_multi_seed,
)

plt.style.use('seaborn-v0_8-whitegrid')
print('PyTorch:', torch.__version__)
print('Dispositivo disponível:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Raiz:', ROOT)

## Parâmetros principais

Os valores abaixo formam um teste CPU razoavelmente rápido. Para um resultado mais forte, use todas as amostras (`train_per_class=None`, `test_per_class=None`), 5–10 épocas e várias seeds. O controle adaptativo usa apenas o split de validação.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'results' / 'split_mnist_notebook'

config = SplitMNISTConfig(
    seed=42,
    hidden_dims=(256, 128),
    batch_size=128,
    epochs_per_task=2,
    train_per_class=1_000,       # None usa todo o treino disponível
    validation_per_class=200,
    test_per_class=500,          # None usa todo o teste
    learning_rate=1e-3,
    weight_decay=1e-4,
    slow_strength=30.0,          # usado nas combinações SlowHeat + baseline
    plasticity_budget=0.25,      # no mínimo 25% dos neurônios livres
    optimizer_state_policy='follow_update',
    adaptive_target_accuracy=0.90,
    adaptive_rate=0.20,
    adaptive_minimum=0.10,
    adaptive_maximum=0.80,
    replay_per_class=20,         # memória: 20 exemplos por classe passada
    replay_batch_size=64,
    distillation_strength=1.0,
    distillation_temperature=2.0,
    methods=(
        'vanilla',
        'slowheat_beta_10',
        'slowheat_beta_30',
        'slowheat_beta_100',
        'hard_freeze',
        'replay',
        'distillation',
        'slowheat_replay',
        'slowheat_distillation',
    ),
    device=DEVICE,
)
config

In [ ]:
tasks = load_split_mnist(config, data_dir=DATA_DIR, download=True)
pd.DataFrame([
    {
        'task': index + 1,
        'classes': str(task.classes),
        'train': len(task.train_y),
        'validation': len(task.validation_y),
        'test': len(task.test_y),
    }
    for index, task in enumerate(tasks)
])

## Comparação principal

- `vanilla`: AdamW sem proteção.
- `slowheat_beta_10/30/100`: proteção suave fatorada em três intensidades.
- `hard_freeze`: congela exatamente todas as unidades consolidadas e suas conexões.
- `replay`: AdamW com memória episódica balanceada.
- `distillation`: AdamW com Learning without Forgetting nos logits antigos.
- `slowheat_replay` e `slowheat_distillation`: combinações com β=30.

In [ ]:
results = run_split_mnist(config, tasks, output_dir=OUTPUT_DIR)

summary = pd.DataFrame([
    {
        'method': method,
        'final_accuracy': result['metrics']['final_average_accuracy'],
        'forgetting': result['metrics']['average_forgetting'],
        'BWT': result['metrics']['backward_transfer'],
        'FWT': result['metrics']['forward_transfer'],
        'task_aware_accuracy': result['task_aware_metrics']['final_average_accuracy'],
        'task_aware_forgetting': result['task_aware_metrics']['average_forgetting'],
        'classifier_gap': result['classifier_gap'],
        'seconds': result['elapsed_seconds'],
    }
    for method, result in results.items()
]).sort_values('final_accuracy', ascending=False)
summary.style.format({
    'final_accuracy': '{:.4f}', 'forgetting': '{:.4f}',
    'BWT': '{:.4f}', 'FWT': '{:.4f}',
    'task_aware_accuracy': '{:.4f}', 'task_aware_forgetting': '{:.4f}',
    'classifier_gap': '{:.4f}', 'seconds': '{:.1f}',
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
stages = np.arange(1, config.task_count + 1)
for method, result in results.items():
    axes[0].plot(stages, result['stage_average_accuracy'], marker='o', label=method)
    axes[1].plot(stages, result['stage_average_forgetting'], marker='o', label=method)
    ci_final = np.array(result['accuracy_matrix'][-1], dtype=float)
    ta_final = np.array(result['task_aware_accuracy_matrix'][-1], dtype=float)
    axes[2].scatter(np.nanmean(ci_final), np.nanmean(ta_final), label=method)
axes[0].set(title='Acurácia média ao longo do stream', xlabel='Tarefas aprendidas', ylabel='Acurácia média', xticks=stages, ylim=(0, 1))
axes[1].set(title='Forgetting acumulado', xlabel='Tarefas aprendidas', ylabel='Forgetting médio', xticks=stages, ylim=(0, 1))
axes[2].plot([0, 1], [0, 1], '--', color='gray')
axes[2].set(title='Diagnóstico do classificador', xlabel='Class-incremental', ylabel='Task-aware', xlim=(0, 1), ylim=(0, 1))
axes[1].legend(loc='best', fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
method_names = list(results)
fig, axes = plt.subplots(1, len(method_names), figsize=(4 * len(method_names), 3.8), squeeze=False)
for axis, method in zip(axes[0], method_names):
    matrix = np.array([[np.nan if value is None else value for value in row] for row in results[method]['accuracy_matrix']])
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    for row in range(matrix.shape[0]):
        for col in range(row + 1):
            axis.text(col, row, f'{matrix[row, col]:.2f}', ha='center', va='center', color='white' if matrix[row, col] < 0.55 else 'black', fontsize=8)
    axis.set(title=method, xlabel='Task avaliada', ylabel='Após task', xticks=range(config.task_count), yticks=range(config.task_count))
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.8, label='Acurácia')
plt.show()

### Onde ocorre o esquecimento?

A diferença entre a avaliação class-incremental e a task-aware é o `classifier_gap`. Gap alto significa que a representação ainda distingue as classes quando o task ID restringe a decisão, mas o classificador global está enviesado/interferindo. Task-aware também baixa significa degradação da representação.

In [ ]:
diagnostic_methods = ['vanilla', 'slowheat_beta_30', 'hard_freeze', 'replay', 'distillation']
fig, axes = plt.subplots(2, len(diagnostic_methods), figsize=(4 * len(diagnostic_methods), 7.5), squeeze=False)
for col, method in enumerate(diagnostic_methods):
    for row, key in enumerate(('accuracy_matrix', 'task_aware_accuracy_matrix')):
        matrix = np.array([[np.nan if value is None else value for value in values] for values in results[method][key]])
        image = axes[row, col].imshow(matrix, vmin=0, vmax=1, cmap='viridis')
        axes[row, col].set(title=f"{method} — {'task-aware' if row else 'class-IL'}", xlabel='Task avaliada', ylabel='Após task')
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.8, label='Acurácia')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for method in ('slowheat_beta_30', 'hard_freeze'):
    if method not in results:
        continue
    axes[0].plot(stages, results[method]['validation_acquisition'], marker='o', label=method)
    history = results[method]['capacity_history']
    plastic = [np.mean([layer['plastic_fraction'] for layer in stage]) for stage in history]
    axes[1].plot(stages[:len(plastic)], plastic, marker='o', label=method)
axes[0].axhline(config.adaptive_target_accuracy, color='black', linestyle='--', label='referência')
axes[0].set(title='Aquisição da tarefa atual (validação)', xlabel='Task', ylabel='Acurácia', xticks=stages, ylim=(0, 1))
axes[1].set(title='Capacidade plástica realizada', xlabel='Task', ylabel='Fração plástica média', xticks=stages, ylim=(0, 1))
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## Sweep dos parâmetros mais importantes

Este sweep usa uma configuração menor para comparar rapidamente força de proteção e capacidade livre. No gráfico de Pareto, o melhor sentido é **direita e baixo**: mais acurácia, menos forgetting.

In [ ]:
RUN_SWEEP = True
SWEEP_STRENGTHS = [10.0, 30.0, 100.0]
SWEEP_BUDGETS = [0.25, 0.50]
SWEEP_EPOCHS = 1
SWEEP_TRAIN_PER_CLASS = 500

sweep_rows = []
if RUN_SWEEP:
    for strength in SWEEP_STRENGTHS:
        for budget in SWEEP_BUDGETS:
            sweep_config = replace(
                config,
                methods=('slowheat',),
                slow_strength=strength,
                plasticity_budget=budget,
                epochs_per_task=SWEEP_EPOCHS,
                train_per_class=SWEEP_TRAIN_PER_CLASS,
            )
            sweep_tasks = load_split_mnist(sweep_config, data_dir=DATA_DIR, download=False)
            run = run_split_mnist(sweep_config, sweep_tasks)['slowheat']
            sweep_rows.append({
                'slow_strength': strength,
                'plasticity_budget': budget,
                'final_accuracy': run['metrics']['final_average_accuracy'],
                'forgetting': run['metrics']['average_forgetting'],
                'BWT': run['metrics']['backward_transfer'],
            })
    sweep = pd.DataFrame(sweep_rows)
    sweep.to_csv(OUTPUT_DIR / 'parameter_sweep.csv', index=False)
else:
    sweep = pd.DataFrame(columns=['slow_strength', 'plasticity_budget', 'final_accuracy', 'forgetting', 'BWT'])
sweep

In [ ]:
if not sweep.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for budget, group in sweep.groupby('plasticity_budget'):
        ordered = group.sort_values('slow_strength')
        axes[0].plot(ordered['slow_strength'], ordered['final_accuracy'], marker='o', label=f'budget={budget}')
        axes[1].plot(ordered['slow_strength'], ordered['forgetting'], marker='o', label=f'budget={budget}')
    axes[0].set(title='Sensibilidade da acurácia', xlabel='slow_strength', ylabel='Acurácia final', ylim=(0, 1))
    axes[1].set(title='Sensibilidade do forgetting', xlabel='slow_strength', ylabel='Forgetting', ylim=(0, 1))
    axes[0].legend(); axes[1].legend()
    for _, row in sweep.iterrows():
        axes[2].scatter(row['final_accuracy'], row['forgetting'], s=60)
        axes[2].annotate(f"β={row['slow_strength']}, p={row['plasticity_budget']}", (row['final_accuracy'], row['forgetting']), xytext=(4, 4), textcoords='offset points', fontsize=8)
    axes[2].set(title='Fronteira estabilidade–plasticidade', xlabel='Acurácia final → melhor', ylabel='Forgetting → melhor')
    fig.tight_layout()
    plt.show()

## Leitura automática do resultado

A célula abaixo compara o método completo com vanilla. Para afirmar melhora global, procure acurácia igual ou maior **e** forgetting menor, com repetição em várias seeds.

In [ ]:
vanilla = results['vanilla']['metrics']
candidate = results['slowheat_beta_30']['metrics']
delta_accuracy = candidate['final_average_accuracy'] - vanilla['final_average_accuracy']
delta_forgetting = candidate['average_forgetting'] - vanilla['average_forgetting']
print(f'Delta de acurácia final: {delta_accuracy:+.4f}')
print(f'Delta de forgetting:      {delta_forgetting:+.4f}  (negativo é melhor)')
if delta_accuracy >= 0 and delta_forgetting <= 0:
    print('Resultado preliminar dominante: melhorou ou preservou ambas as métricas.')
elif delta_forgetting < 0:
    print('Trade-off: houve maior retenção, mas verifique o custo de plasticidade na acurácia.')
else:
    print('Nesta configuração não houve evidência de melhora sobre vanilla.')
print('Artefatos salvos em:', OUTPUT_DIR)

## Execução em dez seeds

Esta célula repete o protocolo completo com seeds pareadas: dentro de cada seed, todos os métodos recebem exatamente a mesma inicialização, partições e minibatches. O CSV final contém média, desvio-padrão e meia-largura do IC95% normal. O JSON inclui diferenças pareadas contra `vanilla` e contra `replay`.

In [ ]:
SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 110]
TEN_SEED_DIR = ROOT / 'results' / 'split_mnist_10seeds'
aggregate = run_split_mnist_multi_seed(
    config,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=TEN_SEED_DIR,
    download=True,
    verbose=True,
)
aggregate_rows = []
for method, metrics in aggregate['methods'].items():
    aggregate_rows.append({
        'method': method,
        'accuracy_mean': metrics['final_average_accuracy']['mean'],
        'accuracy_std': metrics['final_average_accuracy']['std'],
        'forgetting_mean': metrics['average_forgetting']['mean'],
        'forgetting_std': metrics['average_forgetting']['std'],
        'task_aware_mean': metrics['task_aware_final_accuracy']['mean'],
        'classifier_gap_mean': metrics['classifier_gap']['mean'],
    })
aggregate_df = pd.DataFrame(aggregate_rows).sort_values('accuracy_mean', ascending=False)
aggregate_df.style.format(precision=4)

In [ ]:
plot_df = aggregate_df.sort_values('accuracy_mean')
fig, axes = plt.subplots(1, 3, figsize=(19, 5.2))
axes[0].barh(plot_df['method'], plot_df['accuracy_mean'], xerr=plot_df['accuracy_std'], alpha=0.8)
axes[0].set(title='Acurácia final — média ± DP (10 seeds)', xlabel='Acurácia', xlim=(0, 1))
axes[1].barh(plot_df['method'], plot_df['forgetting_mean'], xerr=plot_df['forgetting_std'], alpha=0.8)
axes[1].set(title='Forgetting — média ± DP (10 seeds)', xlabel='Menor é melhor', xlim=(0, 1))
axes[2].scatter(plot_df['accuracy_mean'], plot_df['task_aware_mean'])
for _, row in plot_df.iterrows():
    axes[2].annotate(row['method'], (row['accuracy_mean'], row['task_aware_mean']), xytext=(4, 3), textcoords='offset points', fontsize=8)
axes[2].plot([0, 1], [0, 1], '--', color='gray')
axes[2].set(title='Localização do esquecimento', xlabel='Class-incremental', ylabel='Task-aware', xlim=(0, 1), ylim=(0, 1))
fig.tight_layout()
plt.show()
print('Médias salvas em:', TEN_SEED_DIR / 'aggregate.csv')

### Replay versus SlowHeat + replay

A célula abaixo quantifica automaticamente se adicionar SlowHeat ao replay ajuda. A métrica principal é class-incremental; task-aware é diagnóstica. Se o task-aware continuar alto enquanto o `classifier_gap` aumenta, o gargalo está na competição/recalibração da cabeça classificadora.

In [ ]:
reference = aggregate['methods']['replay']
combined = aggregate['methods']['slowheat_replay']
replay_diagnosis = pd.DataFrame([
    {
        'comparison': 'slowheat_replay − replay',
        'accuracy_delta': combined['final_average_accuracy']['mean'] - reference['final_average_accuracy']['mean'],
        'forgetting_delta': combined['average_forgetting']['mean'] - reference['average_forgetting']['mean'],
        'task_aware_delta': combined['task_aware_final_accuracy']['mean'] - reference['task_aware_final_accuracy']['mean'],
        'classifier_gap_delta': combined['classifier_gap']['mean'] - reference['classifier_gap']['mean'],
        'runtime_ratio': combined['elapsed_seconds']['mean'] / reference['elapsed_seconds']['mean'],
    }
])
display(replay_diagnosis.style.format({
    'accuracy_delta': '{:+.4f}', 'forgetting_delta': '{:+.4f}',
    'task_aware_delta': '{:+.4f}', 'classifier_gap_delta': '{:+.4f}',
    'runtime_ratio': '{:.2f}×',
}))
if replay_diagnosis.iloc[0]['accuracy_delta'] < 0 and replay_diagnosis.iloc[0]['classifier_gap_delta'] > 0:
    print('Diagnóstico: SlowHeat preserva a representação, mas piora a competição no classificador global.')

## O modelo melhora com 1, 5 ou 10 épocas?

Este experimento compara três orçamentos de treino nas mesmas dez seeds. Além do replay vencedor, ele testa SlowHeat + replay sem proteger a camada de saída (`hidden`), β menores (`1, 3, 10`), β=30 para isolar o efeito da saída e orçamentos plásticos de `0.50` e `0.75`. Nomes estruturados codificam a configuração, por exemplo `slowheat_replay_hidden_beta_3_budget_0.50`.

A execução é substancialmente mais longa: são 3 contagens de épocas × 10 seeds × 7 métodos. Os resultados são gravados após cada seed e cada contagem de épocas.

In [ ]:
EPOCH_VALUES = [1, 5, 10]
EPOCH_METHODS = (
    'replay',
    'slowheat_replay',                         # β=30, budget=0.25, protege saída
    'slowheat_replay_hidden_beta_30_budget_0.25',
    'slowheat_replay_hidden_beta_1_budget_0.50',
    'slowheat_replay_hidden_beta_3_budget_0.50',
    'slowheat_replay_hidden_beta_10_budget_0.50',
    'slowheat_replay_hidden_beta_3_budget_0.75',
)
EPOCH_SWEEP_DIR = ROOT / 'results' / 'split_mnist_epoch_sweep'
epoch_config = replace(config, methods=EPOCH_METHODS)
epoch_sweep = run_split_mnist_epoch_sweep(
    epoch_config,
    epochs=EPOCH_VALUES,
    seeds=SEEDS,
    data_dir=DATA_DIR,
    output_dir=EPOCH_SWEEP_DIR,
    download=True,
    verbose=True,
    paired_references=('replay',),
)
epoch_df = pd.read_csv(EPOCH_SWEEP_DIR / 'epoch_sweep.csv')
epoch_df.head()

In [ ]:
METRIC_PANELS = (
    ('final_average_accuracy', 'Acurácia final', 'maior é melhor'),
    ('average_forgetting', 'Forgetting', 'menor é melhor'),
    ('classifier_gap', 'Classifier gap', 'menor é melhor'),
    ('elapsed_seconds', 'Tempo', 'segundos'),
)
fig, axes = plt.subplots(2, 2, figsize=(17, 11), sharex=True)
for axis, (metric, title, ylabel) in zip(axes.ravel(), METRIC_PANELS):
    for method, group in epoch_df.groupby('method', sort=False):
        ordered = group.sort_values('epochs')
        axis.errorbar(
            ordered['epochs'], ordered[f'{metric}_mean'],
            yerr=ordered[f'{metric}_ci95'], marker='o', capsize=4, label=method,
        )
    axis.set(title=f'{title} por orçamento de treino', xlabel='Épocas por tarefa', ylabel=ylabel, xticks=EPOCH_VALUES)
axes[0, 0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.tight_layout()
plt.show()

### Diferenças pareadas contra replay

Cada ponto abaixo usa a diferença dentro da mesma seed antes de calcular média e IC95%. Para acurácia, valores acima de zero favorecem a variante; para forgetting, valores abaixo de zero favorecem a variante.

In [ ]:
paired_rows = []
for epochs in EPOCH_VALUES:
    comparisons = epoch_sweep['results'][str(epochs)]['paired_differences_vs_replay']
    for method, metrics in comparisons.items():
        paired_rows.append({
            'epochs': epochs, 'method': method,
            'accuracy_delta': metrics['final_average_accuracy']['mean'],
            'accuracy_ci95': metrics['final_average_accuracy']['ci95_normal_half_width'],
            'forgetting_delta': metrics['average_forgetting']['mean'],
            'forgetting_ci95': metrics['average_forgetting']['ci95_normal_half_width'],
        })
paired_df = pd.DataFrame(paired_rows)
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), sharex=True)
for method, group in paired_df.groupby('method', sort=False):
    ordered = group.sort_values('epochs')
    axes[0].errorbar(ordered['epochs'], ordered['accuracy_delta'], yerr=ordered['accuracy_ci95'], marker='o', capsize=4, label=method)
    axes[1].errorbar(ordered['epochs'], ordered['forgetting_delta'], yerr=ordered['forgetting_ci95'], marker='o', capsize=4, label=method)
for axis in axes:
    axis.axhline(0, color='black', linestyle='--', linewidth=1)
    axis.set_xlabel('Épocas por tarefa'); axis.set_xticks(EPOCH_VALUES)
axes[0].set(title='Δ acurácia versus replay', ylabel='Positivo favorece a variante')
axes[1].set(title='Δ forgetting versus replay', ylabel='Negativo favorece a variante')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
best_accuracy = epoch_df.loc[epoch_df.groupby('epochs')['final_average_accuracy_mean'].idxmax(), [
    'epochs', 'method', 'final_average_accuracy_mean', 'average_forgetting_mean',
]].rename(columns={'method': 'best_method'})
accuracy_pivot = epoch_df.pivot(index='method', columns='epochs', values='final_average_accuracy_mean')
display(best_accuracy.style.format({'final_average_accuracy_mean': '{:.4f}', 'average_forgetting_mean': '{:.4f}'}))
display(accuracy_pivot.style.format('{:.4f}').highlight_max(axis=1, color='lightgreen'))
print('Na segunda tabela, o destaque mostra a melhor contagem de épocas para cada modelo.')

## Limites do protocolo

Dez seeds e o sweep de épocas tornam a comparação menos frágil, mas Split MNIST continua sendo um benchmark de depuração. Não escolha 10 épocas apenas porque melhora a acurácia: compare também forgetting, classifier gap e custo. Para evidência SOTA, use o conjunto completo e valide em Split CIFAR-100/TinyImageNet contra baselines especializados.